In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1]))   # añade modulo-00-fundamentos a la búsqueda de imports

from utils.spark_session import get_spark

spark = get_spark()
spark.sql("SELECT 1 AS prueba").show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 19:55:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+------+
|prueba|
+------+
|     1|
+------+



# Tema 3 - SQL basico (con spark.sql())

## Bloque 1: Spark desde VS Code

**Spark** es un motor que procesa datos repartiendo el trabajo entre muchas maquinas.

Databricks esta construido sobre el. La **SparkSession** (la variable **spark**) es la puerta de entrada: a traves de ella envias instrucciones a Spark, ya sea en SQL o en Python.

Tu Python normal (variables, print, pandas) corre en el ordenador. Lo que pasa por spark corre en Databricks. 

Pandas funciona con datos que caben en la memoria de una maquina. Spark esta hecho para gigas o teras de datos repartidos. La sintaxis cambia un poco, pero las ideas del Temas 2 son las mismas. 

De pandas a una tabla de SQL puedes consultar.

Tus datos del Tema 2 estan en un CSV en tu computadora. Para consultarlos con SQL en Databricks, hay que enviarlos a Spark y darles un nombre:

In [2]:
import pandas as pd

# 1. Leer en tu computadora, con pandas (como en el Tema 2)
pdf = pd.read_csv("../tema02/datos/observaciones.csv", sep=";", decimal=",")

# 2. Enviar los datos a Spark: ahora es un DataFrame de Spark
df = spark.createDataFrame(pdf)

# 3. Darle un nombre para poder usarlo en SQL
df.createOrReplaceTempView('observaciones')

# 4. Consultar con SQL
resultado = spark.sql("SELECT * FROM observaciones")
resultado.show()

+---------+----------+-----+----------+--------+
|  station|     datum|  tid|temperatur|kvalitet|
+---------+----------+-----+----------+--------+
|Stockholm|2024-11-06|05:00|      -1.4|       G|
|Stockholm|2024-11-06|06:00|      -2.0|       G|
|Stockholm|2024-11-06|07:00|      NULL|       G|
|    Malmö|2024-11-06|05:00|       2.1|       Y|
|    Malmö|2024-11-06|06:00|       2.6|       G|
|    Malmö|2024-11-06|07:00|       3.0|       G|
|   Kiruna|2024-11-06|05:00|     -12.4|       G|
|   Kiruna|2024-11-06|06:00|     -13.1|       G|
|   Kiruna|2024-11-06|06:00|     -13.1|       G|
|   Kiruna|2024-11-06|07:00|     -12.7|       G|
+---------+----------+-----+----------+--------+



**pd.read_csv(...)** -> Leer el CSV  ->  El ordenador (pandas)

**spark.createDataFrame(pdf)**  ->  Envia los datos a Spark -> Databricks

**.createOrReplaceTempView('observaciones')**  ->  Le pone un nombre de tabla -> Databricks

**spark.sql('...')**  ->  Prepara una consulta AQL sobre esa 'tabla' -> Databricks

**.show()** -> Ejecuta la consulta y muestra el resultado  ->  Resulta impreso en tu notebook

**Una vista temporal** (temporary view) es un nombre que le das a un DF para usarlo en SQL. No guarda nada en disco, y existe solo mientras dure tu sesion: si reinicias el kernel, desaparece y hay que volver a crearla. Las tablas permanentes llegaran en el Modulo 1. 

In [3]:
resultado.count()

10

In [4]:
resultado.toPandas()

,station,datum,tid,temperatur,kvalitet
0,Stockholm,2024-11-06,05:00,-1.4,G
1,Stockholm,2024-11-06,06:00,-2.0,G
2,Stockholm,2024-11-06,07:00,NaN,G
3,Malmö,2024-11-06,05:00,2.1,Y
4,Malmö,2024-11-06,06:00,2.6,G
5,Malmö,2024-11-06,07:00,3.0,G
6,Kiruna,2024-11-06,05:00,-12.4,G
7,Kiruna,2024-11-06,06:00,-13.1,G
8,Kiruna,2024-11-06,06:00,-13.1,G
9,Kiruna,2024-11-06,07:00,-12.7,G


In [5]:
resultado.show(5)

+---------+----------+-----+----------+--------+
|  station|     datum|  tid|temperatur|kvalitet|
+---------+----------+-----+----------+--------+
|Stockholm|2024-11-06|05:00|      -1.4|       G|
|Stockholm|2024-11-06|06:00|      -2.0|       G|
|Stockholm|2024-11-06|07:00|      NULL|       G|
|    Malmö|2024-11-06|05:00|       2.1|       Y|
|    Malmö|2024-11-06|06:00|       2.6|       G|
+---------+----------+-----+----------+--------+
only showing top 5 rows


In [6]:
spark.sql("""
    SELECT station, temperatur
    FROM observaciones
""").show()

+---------+----------+
|  station|temperatur|
+---------+----------+
|Stockholm|      -1.4|
|Stockholm|      -2.0|
|Stockholm|      NULL|
|    Malmö|       2.1|
|    Malmö|       2.6|
|    Malmö|       3.0|
|   Kiruna|     -12.4|
|   Kiruna|     -13.1|
|   Kiruna|     -13.1|
|   Kiruna|     -12.7|
+---------+----------+



### Practica guiada - Paso 1

1. Ejecuta los cuatro pasos del ejemplo. 
2. Antes de ejecutar estas lineas, predice:

a - Spark

b - no por que no esta el .show(), lo que imprime es el esquema de la  tabal

c-10 filas

d- ahora sera un pandas core frame datafrmae

In [7]:
   resultado = spark.sql("SELECT * FROM observaciones")
   print(type(resultado))          # a) ¿pandas o Spark?
   print(resultado)                # b) ¿imprime la tabla? ¿qué crees que imprime?
   print(resultado.count())        # c) ¿cuántas filas?
   type(resultado.toPandas())      # d) ¿y ahora qué tipo es?

<class 'pyspark.sql.classic.dataframe.DataFrame'>
DataFrame[station: string, datum: string, tid: string, temperatur: double, kvalitet: string]
10


pandas.core.frame.DataFrame

In [8]:
resultado.show()

spark.sql("""
    SELECT COUNT(*) AS nulos
    FROM observaciones
    WHERE temperatur IS NULL
""").show()

+---------+----------+-----+----------+--------+
|  station|     datum|  tid|temperatur|kvalitet|
+---------+----------+-----+----------+--------+
|Stockholm|2024-11-06|05:00|      -1.4|       G|
|Stockholm|2024-11-06|06:00|      -2.0|       G|
|Stockholm|2024-11-06|07:00|      NULL|       G|
|    Malmö|2024-11-06|05:00|       2.1|       Y|
|    Malmö|2024-11-06|06:00|       2.6|       G|
|    Malmö|2024-11-06|07:00|       3.0|       G|
|   Kiruna|2024-11-06|05:00|     -12.4|       G|
|   Kiruna|2024-11-06|06:00|     -13.1|       G|
|   Kiruna|2024-11-06|06:00|     -13.1|       G|
|   Kiruna|2024-11-06|07:00|     -12.7|       G|
+---------+----------+-----+----------+--------+

+-----+
|nulos|
+-----+
|    1|
+-----+



## Bloque 2: SELECT  

La clausula que dice que columnas quiere en el resultado. Es la base de todo SQL.

Casi nunca necesitas todas las columnas. Elegir solo las necesarias hace la consulta mas clara y, en Spark, **mas rapida**: con datos en Parquet o Delta, Spark lee solo las columnas que pides. Por eso, en codigo de produccion, SELECT * se evita. 

**Elegir columnas y poner alias**

In [9]:
spark.sql("""
    SELECT
        station AS estacion,
        tid AS hora,
        temperatur AS temp_c
    FROM observaciones
""").show()

+---------+-----+------+
| estacion| hora|temp_c|
+---------+-----+------+
|Stockholm|05:00|  -1.4|
|Stockholm|06:00|  -2.0|
|Stockholm|07:00|  NULL|
|    Malmö|05:00|   2.1|
|    Malmö|06:00|   2.6|
|    Malmö|07:00|   3.0|
|   Kiruna|05:00| -12.4|
|   Kiruna|06:00| -13.1|
|   Kiruna|06:00| -13.1|
|   Kiruna|07:00| -12.7|
+---------+-----+------+



**Columnas calculadas**

In [10]:
spark.sql("""
    SELECT
        station,
        temperatur,
        temperatur * 9 / 5 + 32 AS temp_f,
        datum || ' ' || tid AS fecha_hora
    FROM observaciones
""").show()

+---------+----------+-----------------+----------------+
|  station|temperatur|           temp_f|      fecha_hora|
+---------+----------+-----------------+----------------+
|Stockholm|      -1.4|            29.48|2024-11-06 05:00|
|Stockholm|      -2.0|             28.4|2024-11-06 06:00|
|Stockholm|      NULL|             NULL|2024-11-06 07:00|
|    Malmö|       2.1|            35.78|2024-11-06 05:00|
|    Malmö|       2.6|            36.68|2024-11-06 06:00|
|    Malmö|       3.0|             37.4|2024-11-06 07:00|
|   Kiruna|     -12.4|             9.68|2024-11-06 05:00|
|   Kiruna|     -13.1|8.420000000000002|2024-11-06 06:00|
|   Kiruna|     -13.1|8.420000000000002|2024-11-06 06:00|
|   Kiruna|     -12.7|             9.14|2024-11-06 07:00|
+---------+----------+-----------------+----------------+



**DISTINCT** Y **LIMIT**

In [11]:
spark.sql("SELECT DISTINCT station FROM observaciones").show()
# cada estacion una vez

spark.sql("SELECT * FROM observaciones LIMIT 3").show()
# como mucho 3 lineas

+---------+
|  station|
+---------+
|Stockholm|
|    Malmö|
|   Kiruna|
+---------+

+---------+----------+-----+----------+--------+
|  station|     datum|  tid|temperatur|kvalitet|
+---------+----------+-----+----------+--------+
|Stockholm|2024-11-06|05:00|      -1.4|       G|
|Stockholm|2024-11-06|06:00|      -2.0|       G|
|Stockholm|2024-11-06|07:00|      NULL|       G|
+---------+----------+-----+----------+--------+



**DISTINCT** elimina las filas reperidas del resultado. Con varias columnas, SELECT DISTINCT station, kvalitet, cuenta como repetida solo la combinacion completa

**Del Tema 2 al Tema3**

df[["station", "temperatur"]] -> SELECT station, temperatur FROM ...

df.rename(columns={"station": "estacion"}) -> SELECT station AS estacion

df["temp_f"] = df["temperatur"] * 9 / 5 + 32 -> SELECT temperatur * 9 / 5 + 32 AS temp_f

df["station"].unique() -> SELECT DISTINCT station

df.head(3) -> LIMIT 3

### Práctica guiada – Paso 2

Predice primero, sin ejecutar:

1. ¿Cuántas filas devuelve SELECT DISTINCT station FROM observaciones?
2. ¿Cuántas filas devuelve SELECT DISTINCT station, kvalitet FROM observaciones? ¿Por qué no es igual que la 1?
3. Cuántas filas devuelve SELECT DISTINCT * FROM observaciones? Piensa en el Tema 2.
4. En temperatur * 9 / 5 + 32 AS temp_f, ¿qué valor tendrá temp_f para Stockholm 07:00?

Después escribe tú una consulta que muestre estacion, hora y la temperatura en Fahrenheit redondeada a 1 decimal como temp_f. Para redondear, usa la función round(valor, decimales), igual que en Python.

1 -> 3 filas

2 -> 4 filas, por que aqui tambien interviene la columna kvlatitet y exite en malmo una con G y otra con Y

3 -> 10 filas

4 -> tendra null

In [12]:
spark.sql("""
    SELECT
        station AS estacion,
        tid AS hora,
        round(temperatur * 9 / 5 + 32, 1) AS temp_f
    FROM observaciones
""").show()

+---------+-----+------+
| estacion| hora|temp_f|
+---------+-----+------+
|Stockholm|05:00|  29.5|
|Stockholm|06:00|  28.4|
|Stockholm|07:00|  NULL|
|    Malmö|05:00|  35.8|
|    Malmö|06:00|  36.7|
|    Malmö|07:00|  37.4|
|   Kiruna|05:00|   9.7|
|   Kiruna|06:00|   8.4|
|   Kiruna|06:00|   8.4|
|   Kiruna|07:00|   9.1|
+---------+-----+------+



## Bloque 3: WHERE

Filtra las filas que cunplen una condicion. Es la mascara booleana de pandas, en SQL.

Sirve para limpiar(solo calidad Gold), centrarse en una parte de los datos(solo Kiruna, solo noviembre), o encontrarse problemas (filas con nulos).

En Spark, ademas, filtrar pronto hace que todo lo demas procese menos.


In [13]:
spark.sql("""
    SELECT station, tid, temperatur
    FROM observaciones
    WHERE kvalitet = 'G' AND temperatur < 0
""").show()

+---------+-----+----------+
|  station|  tid|temperatur|
+---------+-----+----------+
|Stockholm|05:00|      -1.4|
|Stockholm|06:00|      -2.0|
|   Kiruna|05:00|     -12.4|
|   Kiruna|06:00|     -13.1|
|   Kiruna|06:00|     -13.1|
|   Kiruna|07:00|     -12.7|
+---------+-----+----------+



In [14]:
spark.sql("""
    SELECT *
    FROM observaciones
    WHERE station = :estacion""",
    args={"estacion": "Kiruna"}
).show()

+-------+----------+-----+----------+--------+
|station|     datum|  tid|temperatur|kvalitet|
+-------+----------+-----+----------+--------+
| Kiruna|2024-11-06|05:00|     -12.4|       G|
| Kiruna|2024-11-06|06:00|     -13.1|       G|
| Kiruna|2024-11-06|06:00|     -13.1|       G|
| Kiruna|2024-11-06|07:00|     -12.7|       G|
+-------+----------+-----+----------+--------+



### Practica guiada - Paso 3

Predice cuantas filas devuelve WHERE sin ejecutar

a) WHERE kvalitet = 'G'

b) WHERE temperatur < 0

c) WHERE NOT temperatur < 0

d) WHERE station IN ('Malmö', 'Kiruna') AND kvalitet = 'G'

e) WHERE temperatur BETWEEN -13 AND -12

f) WHERE station LIKE 'K%'

Para la b y la c, piensa en la fila de Stockholm 07:00: ¿pasa en alguna de las dos? ¿Suman 10 entre las dos?

Después escribe tú una consulta que encuentre las filas problemáticas: las que tienen la temperatura nula o una calidad distinta de 'G'. Muestra station, tid, temperatur y kvalitet.

a -> 9 filas

b -> 6 filas

c -> 3 filas

d -> 6 filas

e -> 2 filas

f -> 4 filas

In [15]:
spark.sql("""
    SELECT station, tid, temperatur, kvalitet
    FROM observaciones
    WHERE temperatur IS NULL OR kvalitet != 'G'
""").show()

+---------+-----+----------+--------+
|  station|  tid|temperatur|kvalitet|
+---------+-----+----------+--------+
|Stockholm|07:00|      NULL|       G|
|    Malmö|05:00|       2.1|       Y|
+---------+-----+----------+--------+



## Bloque 4: ORDER BY

Ordena la filas del resultado. Es el sort_values de pandas.

In [16]:
spark.sql("""
    SELECT station, tid, temperatur
    FROM observaciones
    ORDER BY temperatur ASC
""").show()

+---------+-----+----------+
|  station|  tid|temperatur|
+---------+-----+----------+
|Stockholm|07:00|      NULL|
|   Kiruna|06:00|     -13.1|
|   Kiruna|06:00|     -13.1|
|   Kiruna|07:00|     -12.7|
|   Kiruna|05:00|     -12.4|
|Stockholm|06:00|      -2.0|
|Stockholm|05:00|      -1.4|
|    Malmö|05:00|       2.1|
|    Malmö|06:00|       2.6|
|    Malmö|07:00|       3.0|
+---------+-----+----------+



- **ASC** (ascendente, de menor a mayor) es el valor por defecto, asi que se puede omitir.
- **DESC**: de mayor a menor.
- Con varias columnas, se ordena por la primera, y las siguientes solo desempatan. 
- Los textos se ordenan alfabeticamente

**El orden en que SQL ejecuta las clausulas**

Esto explica muchas reglas de SQL. Escribes las clausalas en un orden, pero SQL las ejecuta en otro:

| Orden de escritura | 	Orden de ejecución |
| --- | --- |
| 1. SELECT | 1. FROM: de qué tabla |
| 2. FROM | 2. WHERE: qué filas |
| 3. WHERE | 3. GROUP BY: cómo agrupar |
| 4. GROUP BY | 4. HAVING: qué grupos |
| 5. HAVING | 5. SELECT: qué columnas y alias |
| 6. ORDER BY | 6. ORDER BY: en qué orden |
| 7. LIMIT | 7. LIMIT: cuántas filas |

Por eso :
- En el Where no puedes usar un alias del SELECT: el WHERE se ejcuta antes, y el alias todavia no existe.

- En el ORDER BY si puedes: se ejecuta despues del SELECT.

- ORDER BY + LIMIT dan los "N primeros" de verdad: primero se ordena, despues se corta. 

### Practica guiada - Paso 4

Predice, sin ejecutar:

1. Con ORDER BY temperatur (ascendente), ¿qué estación y hora aparecen en la primera fila? ¿Y en la segunda?
2. Con ORDER BY temperatur DESC, ¿qué hay en la última fila?
3. Con ORDER BY station, tid DESC, ¿cuál es la primera fila?

Después escribe tú una consulta con las 3 temperaturas más frías, solo de mediciones con calidad G y sin nulos, mostrando station, tid y la temperatura con el alias temp_c. Ordena usando el alias, no el nombre original de la columna.

1 -> estocolmo 07:00 kiruna 06:00

2 -> Malmö 07

3 -> Kiruna 05

In [17]:
spark.sql("""
    SELECT station, tid, temperatur AS temp_c
    FROM observaciones
    WHERE kvalitet = :g
    AND temperatur IS NOT NULL
    ORDER BY temp_c ASC
    LIMIT 3
""", args={"g": "G"}).show()

+-------+-----+------+
|station|  tid|temp_c|
+-------+-----+------+
| Kiruna|06:00| -13.1|
| Kiruna|06:00| -13.1|
| Kiruna|07:00| -12.7|
+-------+-----+------+



## Bloque 5: agregaciones y GROUP BY

Resumir muchas filas en una sola (agregar), ya sea toda la tabla o por grupos. Es el groupby().agg() de pandas.

**Las funciones de agregacion**

In [18]:
spark.sql("""
    SELECT
        COUNT(*)               AS filas,
        COUNT(temperatur)      AS con_temperatura,
        COUNT(DISTINCT station) AS estaciones,
        MIN(temperatur)        AS minima,
        MAX(temperatur)        AS maxima,
        ROUND(AVG(temperatur), 1) AS media,
        SUM(temperatur)        AS suma
    FROM observaciones
""").show()

+-----+---------------+----------+------+------+-----+-----+
|filas|con_temperatura|estaciones|minima|maxima|media| suma|
+-----+---------------+----------+------+------+-----+-----+
|   10|              9|         3| -13.1|   3.0| -5.2|-47.0|
+-----+---------------+----------+------+------+-----+-----+



| Funcion | Que hace | pandas |
| --- | --- | --- |
| COUNT(*) | Cuenta filas, incluidas las que tienen NULL | size |
| COUNT(DISTINCT columna) | Cuenta los valores distintos | nunique |
| SUM,AVG,MIN,MAX | Suma, Media, Minima, Maximo | sum,mean,min,maz |
| COUNT(columna) | Cunta los valores no  ulos de esa columna | count |

Las agregaciones ignoran los NULL (salvo COUNT(*)). AVG no cuneta el NULL como 0: lo deja fuera, y divide entre los valores que si existen. Para Stockholm, AVG es (-1.4 + -2.0) / 2, no /3

**Por grupos: GROUP BY**

In [19]:
spark.sql("""
    SELECT
        station,
        COUNT(*) AS n,
        ROUND(AVG(temperatur), 1) AS media
    FROM observaciones
    GROUP BY station
""").show()

+---------+---+-----+
|  station|  n|media|
+---------+---+-----+
|Stockholm|  3| -1.7|
|    Malmö|  3|  2.6|
|   Kiruna|  4|-12.8|
+---------+---+-----+



La regla de oro del GROUP BY

Toda columna del SELECT tiene que estar en el GROUP BY, o dentro de una funcion de agregacion. 

In [20]:
# SELECT station, tid, AVG(temperatur)   -- ❌ error
# FROM observaciones
# GROUP BY station

Este cogigo **FALLA** por que el grupo de KIRUNA tiene cuatro valores de tid ditintos, y SQL no sabe cual mostrar en la unica fila de KIRUNA.

**Para el examen -> approx_count_distincs:**

In [21]:
# SELECT approx_count_distinct(station) FROM observaciones

Con miles de millones de filas, contar exactamente los valores distintos es caro , porque Spark tiene que comparar todos con todos. 

**approx_count_distinct** da una aproximacion muy cercana y mucho mas rapida. Aparece literalmente en la guia del examen de Data ENgineer

### Practica guiada - Paso 5

Predice, sin ejecutar, los valores de la consulta de toda la tabla (la primera de los ejemplos): filas, con_temperatura, estaciones y minima.

Después escribe tú una consulta con una fila por estación y estas columnas:

- n_mediciones: todas las filas
- n_validas: las que tienen temperatura
- minima, maxima
- media, redondeada a 1 decimal

Ordena de la estación con la media más fría a la más cálida.

Antes de ejecutarla, calcula a mano la media de Kiruna y la de Stockholm. Piensa: ¿la fila duplicada cuenta en Kiruna? ¿Y el NULL en Stockholm?

In [22]:
spark.sql("""
    SELECT
        station,
        COUNT(*) AS n_mediciones,
        COUNT(temperatur) AS n_validas,
        MIN(temperatur) AS minima,
        MAX(temperatur) AS maxima,
        ROUND(AVG(temperatur), 1) AS media
    FROM observaciones
    GROUP BY station
    ORDER BY media
""").show()

+---------+------------+---------+------+------+-----+
|  station|n_mediciones|n_validas|minima|maxima|media|
+---------+------------+---------+------+------+-----+
|   Kiruna|           4|        4| -13.1| -12.4|-12.8|
|Stockholm|           3|        2|  -2.0|  -1.4| -1.7|
|    Malmö|           3|        3|   2.1|   3.0|  2.6|
+---------+------------+---------+------+------+-----+



## Bloque 6: HAVING

Filtrar grupos, despues de agregar. WHERE filtra filas, antes de agregar.

In [23]:
spark.sql("""
    SELECT station, ROUND(AVG(temperatur), 1) AS media
    FROM observaciones
    WHERE kvalitet = 'G'               -- 1. filas: solo calidad G
    GROUP BY station                   -- 2. agrupar
    HAVING AVG(temperatur) < 0         -- 3. grupos: solo los de media negativa
""").show()

+---------+-----+
|  station|media|
+---------+-----+
|Stockholm| -1.7|
|   Kiruna|-12.8|
+---------+-----+



La diferencia, con el order de ejecucion del Bloque 4:

|  | WHERE | Having |
| --- | --- | --- |
| Cuando se ejecuta | Antes del GROUP BY | Despues del GROUP BY |
| Que filtra | Filas sueltas | Grupos enteros |
| Puede usar agregaciones(AVG, COUNT...) | ❌ No: todavía no hay grupos | ✅ Sí |
| Pregunta tipica | “¿Qué mediciones uso?” | “¿Qué estaciones muestro?” |

En pandas, WHERE es el filtro antes del groupby, y Having es el filtro sobre el resultado del groupby. 

⚠️ Trampas típicas:

- Agregaciones en el WHERE: WHERE AVG(temperatur) < 0 da error, porque en el WHERE todavía no hay grupos.
- Filtrar en HAVING lo que podría ir en WHERE: HAVING station = 'Kiruna' funciona, pero agrupa todas las estaciones para después tirar casi todas. Con WHERE station = 'Kiruna', Spark descarta las filas antes de agrupar, y trabaja con muchos menos datos. Regla: si no necesita una agregación, va en el WHERE.
- El alias en el HAVING: Spark acepta HAVING media < 0, pero muchos otros motores SQL no. Repetir la expresión (HAVING AVG(temperatur) < 0) funciona en todos.

### Práctica guiada – Paso 6

Predice qué estaciones devuelve cada consulta:

a) WHERE kvalitet = 'G'  GROUP BY station  HAVING AVG(temperatur) < 0

b) GROUP BY station  HAVING COUNT(*) > 3

c) GROUP BY station  HAVING COUNT(temperatur) = COUNT(*)

Para la c, piensa qué significa que los dos conteos sean iguales.

Después escribe tú una consulta que muestre las estaciones con media negativa y al menos 3 mediciones válidas, contando solo las de calidad G. Muestra station, n_validas y media (1 decimal). Piensa qué condición va en el WHERE y cuál en el HAVING.

a -> Stockholm : -1.7
    Kiruna : -12.8

b -> Kiruna : 4

c -> Kiruna 
    Malmö

In [24]:
spark.sql("""
    SELECT
        station,
        COUNT(*) AS n_mediciones,
        COUNT(temperatur) AS n_validas,
        ROUND(AVG(temperatur), 1) AS media
    FROM observaciones
    WHERE kvalitet = :g
    GROUP BY station
    HAVING AVG(temperatur) < 0
        AND COUNT(temperatur) >= 3
""", args={"g": "G"}).show()

+-------+------------+---------+-----+
|station|n_mediciones|n_validas|media|
+-------+------------+---------+-----+
| Kiruna|           4|        4|-12.8|
+-------+------------+---------+-----+



## Bloque 7: expresiones utiles

CASE WHEN: condiciones en una columna

Es el np.where de pandas, pero con tantas ramas como quieras

In [25]:
spark.sql("""
    SELECT
        station,
        tid,
        temperatur,
        CASE
            WHEN temperatur IS NULL THEN 'sin dato'
            WHEN temperatur <= -10  THEN 'muy frío'
            WHEN temperatur <= 0    THEN 'helada'
            ELSE 'ok'
        END AS estado
    FROM observaciones
""").show()

+---------+-----+----------+--------+
|  station|  tid|temperatur|  estado|
+---------+-----+----------+--------+
|Stockholm|05:00|      -1.4|  helada|
|Stockholm|06:00|      -2.0|  helada|
|Stockholm|07:00|      NULL|sin dato|
|    Malmö|05:00|       2.1|      ok|
|    Malmö|06:00|       2.6|      ok|
|    Malmö|07:00|       3.0|      ok|
|   Kiruna|05:00|     -12.4|muy frío|
|   Kiruna|06:00|     -13.1|muy frío|
|   Kiruna|06:00|     -13.1|muy frío|
|   Kiruna|07:00|     -12.7|muy frío|
+---------+-----+----------+--------+



**COALESCE**: reemplazar los NULL

Devuelve el primer valor que no sea NULL, de izq a derecha:

In [26]:
# COALESCE(temperatur, 0)          -- si es NULL, 0
# COALESCE(kvalitet, 'desconocida') -- si es NULL, 'desconocida'

Es el fillna de pandas. Y resuelve lo del Bloque 3: **COALESCE(kvalitet, '') != 'G'** tambien detecta las calidades nulas, porque primera las convierte en ''.

**Funciones de texto**

| Funcion | Ejemplo | Resultado |
| --- | --- | --- |
| UPPER / LOWER | UPPER(station) | 'KIRUNA' |
| TRIM | TRIM('    Lund    ') | 'Lund' |
| LENGTH | LENGTH(station) | 6 |
| SUBSTRING(texto, inicio, largo) | SUBSTRING(tid, 1, 2) | '05'. Ojo: en SQL se empieza a contar en 1, no en 0 |
| REPLACE | REPLACE('2,5', ',', '.') | '2.5' |

**Funciones de fecha**

In [27]:
# to_date(datum)  -- texto → fecha
# to_timestamp(datum || ' ' || tid, 'yyyy-MM-dd HH:mm')
# -- texto → fecha y hora

# HOUR(ts), MONTH(ts), YEAR(ts)  -- extraer partes
# date_format(ts, 'HH:mm')   -- fecha → texto con formato

# timestamp_millis(1730865600000)
# -- epoch en ms → fecha y hora (lo de SMHI)

### Práctica guiada – Paso 7

Predice primero, para la consulta del CASE WHEN de arriba:

1.  ¿Qué estado tienen Stockholm 07:00, Malmö 05:00, Kiruna 05:00 y Stockholm 06:00?
2. Si quitaras la rama WHEN temperatur IS NULL, ¿qué estado tendría Stockholm 07:00?

Después escribe tú una consulta con estas columnas:

- estacion: el nombre de la estación en mayúsculas
- fecha_hora: datum y tid convertidos a fecha y hora con to_timestamp
- hora: solo la hora como número, sacada de fecha_hora
- temp_c: la temperatura, con los NULL reemplazados por el texto… ¡no, espera! Piensa: ¿se puede poner un texto en una columna numérica? Mejor: temp_c, con los NULL sin tocar
- estado: el CASE WHEN de arriba

1 -> sin dato, ok, muy frio, helada

2 -> ok

In [28]:
spark.sql("""
    SELECT
        UPPER(station) AS estacion,
        to_timestamp(CONCAT(datum, ' ', tid), 'yyyy-MM-dd HH:mm') AS fecha_hora,
        HOUR(to_timestamp(CONCAT(datum, ' ', tid), 'yyyy-MM-dd HH:mm')) AS hora,
        temperatur AS temp_c,
        CASE
            WHEN temperatur IS NULL THEN 'sin dato'
            WHEN temperatur <= -10 THEN 'muy frío'
            WHEN temperatur <= 0 THEN 'helada'
            ELSE 'ok'
        END AS estado
    FROM observaciones
""").show()

+---------+-------------------+----+------+--------+
| estacion|         fecha_hora|hora|temp_c|  estado|
+---------+-------------------+----+------+--------+
|STOCKHOLM|2024-11-06 05:00:00|   5|  -1.4|  helada|
|STOCKHOLM|2024-11-06 06:00:00|   6|  -2.0|  helada|
|STOCKHOLM|2024-11-06 07:00:00|   7|  NULL|sin dato|
|    MALMÖ|2024-11-06 05:00:00|   5|   2.1|      ok|
|    MALMÖ|2024-11-06 06:00:00|   6|   2.6|      ok|
|    MALMÖ|2024-11-06 07:00:00|   7|   3.0|      ok|
|   KIRUNA|2024-11-06 05:00:00|   5| -12.4|muy frío|
|   KIRUNA|2024-11-06 06:00:00|   6| -13.1|muy frío|
|   KIRUNA|2024-11-06 06:00:00|   6| -13.1|muy frío|
|   KIRUNA|2024-11-06 07:00:00|   7| -12.7|muy frío|
+---------+-------------------+----+------+--------+

